In [11]:
using MyPackage
using MyPackage.Geometry
using MyPackage.VLM

using Printf

function new_plane()
    plain = Airfoil("../assets/airfoils/Plain/Plain.dat")
    wing = Surface(
        airfoils=[plain, plain],
        b=6.0,
        chord=y -> 2 * (1 - y^2)^0.5,
        sw_center=0.5,
    )
    return Plane([wing])
end

function new_plane2()
    s1223 = Airfoil("../assets/airfoils/S1223/S1223.dat")
    wing = wing = Surface(
        airfoils=[s1223, s1223],
        b=0.850,
        chord=y -> 0.2,
        sw_center=0.5,
    )
    return Plane([wing])
end

function run_example()
    println("\n" * "="^72)
    println("VLM Solver Example")
    println("="^72)

    V_mag = 10.0
    alpha_deg = -5.0
    beta_deg = 0.0
    rho = 1.225
    mu = 1.81 * 10^(-5)
    S_ref = 0.85*0.2
    # S_ref = 3 * pi
    q_inf = 0.5 * rho * V_mag^2

    plane = new_plane2()

    println("\nFreestream: V = $(V_mag) m/s, alpha = $(alpha_deg)°, beta = $(beta_deg)°")
    println("Reference area: $(S_ref) m²")
    println("Q inf: $(q_inf)")
    println("CG: $(plane.data.CG)")

    t0 = time()
    FX, FX_dist, FY, FY_dist, FZ, FZ_dist, L, L_dist, D, D_dist, M, M_dist, Ml, Ml_dist, N, N_dist, L_trefftz, L_dist_trefftz, D_trefftz, D_dist_trefftz = VLMSolver(
        plane,
        V_mag,
        (alpha_deg, beta_deg);
        n_chordxspan=[(30, 70)],
        rho=rho,
    )
    elapsed = time() - t0

    CL = L[1] / (q_inf * S_ref)
    CD = D[1] / (q_inf * S_ref)
    CL_trefftz = L_trefftz[1] / (q_inf * S_ref)
    CD_trefftz = D_trefftz[1] / (q_inf * S_ref)

    Re = rho * V_mag * plane.surfaces[1].MAC / mu

    println("\nResults")
    println("-"^72)
    @printf("Solve time       : %.3f s\n", elapsed)
    @printf("Reynolds number  : %.1f\n", Re)
    @printf("Lift             : %.6f N\n", L[1])
    @printf("Drag             : %.6f N\n", D[1])
    @printf("CL               : %.6f\n", CL)
    @printf("CD               : %.6f\n", CD)
    @printf("Trefftz lift     : %.6f N\n", L_trefftz[1])
    @printf("Trefftz drag     : %.6f N\n", D_trefftz[1])
    @printf("Trefftz CL       : %.6f N\n", CL_trefftz)
    @printf("Trefftz CD       : %.6f N\n", CD_trefftz)

    println("\nSummary")
    @printf("%-10s  %-12s  %-12s\n", "Quantity", "Value", "Units")
    @printf("%-10s  %-12.6f  %-12s\n", "CL", CL, "-")
    @printf("%-10s  %-12.6f  %-12s\n", "CD", CD, "-")
    @printf("%-10s  %-12.6f  %-12s\n", "L", L[1], "N")
    @printf("%-10s  %-12.6f  %-12s\n", "D", D[1], "N")

    return FX, FX_dist, FY, FY_dist, FZ, FZ_dist, L, L_dist, D, D_dist, M, M_dist, Ml, Ml_dist, N, N_dist, L_trefftz, L_dist_trefftz, D_trefftz, D_dist_trefftz
end

result = run_example()


VLM Solver Example
Loading airfoil: S1223 from ../assets/airfoils/S1223/S1223.dat

Freestream: V = 10.0 m/s, alpha = -5.0°, beta = 0.0°
Reference area: 0.17 m²
Q inf: 61.25000000000001
CG: (0.05000000000000001, 0.0, 0.0)

Results
------------------------------------------------------------------------
Solve time       : 3.598 s
Reynolds number  : 135359.1
Lift             : -5.919163 N
Drag             : -1.185437 N
CL               : -0.568467
CD               : -0.113847
Trefftz lift     : -6.350255 N
Trefftz drag     : 0.301952 N
Trefftz CL       : -0.609868 N
Trefftz CD       : 0.028999 N

Summary
Quantity    Value         Units       
CL          -0.568467     -           
CD          -0.113847     -           
L           -5.919163     N           
D           -1.185437     N           


([-1.6968148927303561], [[0.015633688963948664, 0.0009187011593560662, -8.004770640088333e-5, -0.0007968190670759154, -0.001538251100823625, -0.0023504157163627432, -0.003231800997211399, -0.004169452717231933, -0.005147997350313728, -0.006152304112486382  …  -0.0061487214198167744, -0.005143710458640534, -0.0041644173465227915, -0.003226058293707585, -0.002344210923537477, -0.001532326471461944, -0.0007933201819811287, -8.701574555690802e-5, 0.0008075119020351699, 0.009206494218051166]], [7.84940658719547e-17], [[-3.501951841304497e-17, -2.6890450574016865e-18, -8.725113012610157e-18, -9.67576323560428e-18, -1.1410234586930383e-17, 3.305709593723397e-17, -1.2131279985958975e-17, -1.519339843023213e-17, 1.2473741693106544e-17, 1.6636443753357176e-17  …  2.333094350569092e-18, 1.559485424507668e-17, -1.2112003038091794e-17, -1.0523689537920137e-18, 2.579638245292931e-17, -9.253778038727088e-18, -3.150265085093494e-18, 5.812591546965224e-18, 2.4424622521018318e-17, 9.259484923209214e-18]